In [0]:
dbutils.widgets.text("catalog_name", "dbr_dev_ua5816bd", "Catalog Name")
dbutils.widgets.text("silver_schema", "oles0305_silver", "Silver Schema")

CATALOG_NAME = dbutils.widgets.get("catalog_name")
SILVER_SCHEMA = dbutils.widgets.get("silver_schema")

SILVER = f"{CATALOG_NAME}.{SILVER_SCHEMA}"

In [0]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {SILVER}.drivers (
        driverId INT, 
        driverRef STRING, 
        number INT, 
        code STRING, 
        forename STRING, 
        surname STRING, 
        dob DATE, 
        nationality STRING, 
        url STRING,
        bronze_ingested_at TIMESTAMP,
        silver_processed_at TIMESTAMP,
        valid_from TIMESTAMP, 
        valid_to TIMESTAMP, 
        is_active BOOLEAN
    )
    USING DELTA
    TBLPROPERTIES (
        'delta.columnMapping.mode' = 'name'
    )
""")

In [0]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {SILVER}.results (
        resultId INT, 
        raceId INT, 
        driverId INT, 
        constructorId INT,
        grid INT, 
        position INT, 
        points DOUBLE, 
        laps INT, 
        milliseconds INT,
        statusId INT, 
        time STRING,
        bronze_ingested_at TIMESTAMP,
        silver_processed_at TIMESTAMP
    )
    USING DELTA
    CLUSTER BY (raceId, driverId) -- Liquid Clustering
    TBLPROPERTIES (
        'delta.enableTypeWidening' = 'true'
    )
""")

In [0]:
spark.sql(f"ALTER TABLE {SILVER}.results ADD CONSTRAINT valid_points CHECK (points >= 0)")
spark.sql(f"ALTER TABLE {SILVER}.results ADD CONSTRAINT valid_ids CHECK (driverId IS NOT NULL AND raceId IS NOT NULL)")

In [0]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {SILVER}.sprint_results (
        resultId INT, 
        raceId INT, 
        driverId INT, 
        constructorId INT,
        grid INT, 
        position INT, 
        points DOUBLE, 
        laps INT, 
        time STRING,
        milliseconds INT,
        statusId INT, 
        evh_ingested_at TIMESTAMP,
        silver_processed_at TIMESTAMP
    )
    USING DELTA
    CLUSTER BY (raceId, resultId) -- Liquid Clustering
""")

In [0]:
spark.sql(f"ALTER TABLE {SILVER}.sprint_results ADD CONSTRAINT valid_points CHECK (points >= 0)")
spark.sql(f"ALTER TABLE {SILVER}.sprint_results ADD CONSTRAINT valid_ids CHECK (driverId IS NOT NULL AND raceId IS NOT NULL)")